# Experiment 8 — Time homogeneity

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** Check that the selected policy model coefficients don't
drift across negotiation rounds. Fit the model on three time buckets
(early 3–5, middle 6–8, late 9–12) and compare coefficients.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression

TIME_WINDOWS = {'early_3_5': (3, 5), 'middle_6_8': (6, 8), 'late_9_12': (9, 12)}

def fit_window(policy, lo, hi):
    feat, mdl, alpha = SELECTED[policy]
    meta, X, y, cols = nu.build_design_matrix(turns, policy, feat, round_min=lo, round_max=hi)
    if len(X) < 30: return None
    scaler = StandardScaler().fit(X)
    y_logit = nu.logit_clip(y)
    if mdl == 'OLS':
        reg = LinearRegression().fit(scaler.transform(X), y_logit)
    else:
        a = float(alpha) if np.isfinite(alpha) else 1.0
        reg = Ridge(alpha=a).fit(scaler.transform(X), y_logit)
    return {'cols': cols, 'coef': reg.coef_, 'n': len(X)}

records = []
for p in POLICIES:
    for wname, (lo, hi) in TIME_WINDOWS.items():
        info = fit_window(p, lo, hi)
        if info is None: continue
        for c, beta in zip(info['cols'], info['coef']):
            records.append({'policy': p, 'window': wname, 'term': c, 'coef': beta})

tw_df = pd.DataFrame(records)
for p in POLICIES[:2]:
    print(f'\n=== {p}: coefficients by time window ===')
    print(tw_df[tw_df.policy == p]
           .pivot(index='term', columns='window', values='coef').round(3))